&nbsp;

&nbsp;

## [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) Result Pre-process Notebook

- [./encode_re2g_parse.ipynb](./encode_re2g_parse.ipynb)に必要なデータを揃えるための notebook
- 忘れないように、明示的にこちらで処理する

In [3]:
import os
import pathlib
import polars as pl
from IPython.display import display

os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## Parse `bedpe` file (Ctrl)

- [ENCODE portal](https://www.encodeproject.org/matrix/?type=Annotation&status=released&annotation_type=element+gene+regulatory+interaction+predictions&software_used.software.name=distal-regulation-encode_re2g&assembly=GRCh38&files.file_type=bedpe) から取得した `bedpe`ファイル (ABC model と ENCODE-rE2G model の結果 両方の bedpe ファイルがあるが、今回は ENCODE-rE2G のデータを使用する)
- [An encyclopedia of human enhancer–gene regulatory interactions.](https://doi.org/10.1038/s41586-026-10781-4) の Supplementary Table 12 の全ての予測リストと合わせて、取得済み
- enhancer-gene の関係性は失われてしまうが、fanta.bio でのシスエレメントの重なりを評価するため、enhancer の情報だけの`BED`ファイルにする

In [4]:
#test
ENCSR000MZA_ENCFF654AMU = pl.read_csv(
    "../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_bedpe/ENCSR000MZA_ENCFF654AMU.bedpe.gz",
    separator="\t",
    has_header=False,
    n_threads=16
)

display(ENCSR000MZA_ENCFF654AMU)

column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10
str,i64,i64,str,i64,i64,str,f64,str,str
"""chr1""",778470,778970,"""chr1""",827522,827523,"""LINC00115|chr1:778470-778970""",0.68826,""".""","""."""
"""chr1""",778470,778970,"""chr1""",827590,827591,"""LINC01128|chr1:778470-778970""",0.688063,""".""","""."""
"""chr1""",827272,827840,"""chr1""",827522,827523,"""LINC00115|chr1:827272-827840""",0.999996,""".""","""."""
"""chr1""",827272,827840,"""chr1""",827590,827591,"""LINC01128|chr1:827272-827840""",0.999996,""".""","""."""
"""chr1""",869627,870127,"""chr1""",827522,827523,"""LINC00115|chr1:869627-870127""",0.361537,""".""","""."""
…,…,…,…,…,…,…,…,…,…
"""chrX""",155997330,155998044,"""chrX""",155997580,155997581,"""IL9R|chrX:155997330-155998044""",0.999994,""".""","""."""
"""chrX""",155998888,155999820,"""chrX""",155997580,155997581,"""IL9R|chrX:155998888-155999820""",0.979845,""".""","""."""
"""chrX""",156000372,156000872,"""chrX""",155997580,155997581,"""IL9R|chrX:156000372-156000872""",0.957969,""".""","""."""


In [7]:
# Input and output directories
bedpe_dir = pathlib.Path("../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_bedpe")
out_dir = pathlib.Path("../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_only_enhancer")
out_dir.mkdir(parents=True, exist_ok=True)

# Collect all .bedpe.gz files (expected: 1460 files)
bedpe_paths = sorted(bedpe_dir.glob("*.bedpe.gz"))
print(f"found: {len(bedpe_paths)} files")

for bedpe_path in bedpe_paths:
    # e.g. ENCSR000MZA_ENCFF654AMU.bedpe.gz -> ENCSR000MZA_ENCFF654AMU_only_enhancer.bed
    sample_name = bedpe_path.name.removesuffix(".bedpe.gz")
    out_path = out_dir / f"{sample_name}_only_enhancer.bed"

    # Read only the first 3 columns (enhancer chrom, start, end)
    bedpe = pl.read_csv(
        bedpe_path,
        separator="\t",
        has_header=False,
        columns=[0, 1, 2],
    )

    # Remove duplicated enhancers and sort by position
    enhancer_bed = bedpe.unique().sort("column_1", "column_2", "column_3")

    enhancer_bed.write_csv(
        out_path,
        separator="\t",
        include_header=False,
    )

print(f"done: {len(list(out_dir.glob('*_only_enhancer.bed')))} files written to {out_dir}")

found: 1460 files
done: 1460 files written to ../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_only_enhancer


&nbsp;

&nbsp;

&nbsp;

## Parse `bedpe` file (HS) (COMPLETE)

- enhancer-gene の関係性は失われてしまうが、enhancer の情報だけの`BED`ファイルにする

In [6]:
# test
GSE101731_HFF_44C_60min_atacseq = pl.read_csv(
    "../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE101731_HFF_44C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe",
    separator="\t",
    has_header=False,
    n_threads=16
)

display(GSE101731_HFF_44C_60min_atacseq)


column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10
str,i64,i64,str,i64,i64,str,f64,str,str
"""chr1""",778460,779061,"""chr1""",827590,827590,"""LINC01128_intergenic|chr1:7784…",0.327715,""".""","""."""
"""chr1""",778460,779061,"""chr1""",827522,827522,"""LINC00115_intergenic|chr1:7784…",0.326647,""".""","""."""
"""chr1""",820689,821475,"""chr1""",827590,827590,"""LINC01128_intergenic|chr1:8206…",0.599405,""".""","""."""
"""chr1""",820689,821475,"""chr1""",827522,827522,"""LINC00115_intergenic|chr1:8206…",0.600689,""".""","""."""
"""chr1""",827239,827840,"""chr1""",827590,827590,"""LINC01128_promoter|chr1:827239…",0.999969,""".""","""."""
…,…,…,…,…,…,…,…,…,…
"""chrX""",155070900,155072184,"""chrX""",155071419,155071419,"""BRCC3_promoter|chrX:155070900-…",0.999622,""".""","""."""
"""chrX""",155215781,155217471,"""chrX""",155216423,155216423,"""VBP1_promoter|chrX:155215781-1…",0.998998,""".""","""."""
"""chrX""",155264254,155264839,"""chrX""",155264589,155264589,"""RAB39B_promoter|chrX:155264254…",0.999976,""".""","""."""


In [2]:
# # Specify input .bedpe paths here
# bedpe_paths = [
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE101731_HFF_44C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE101731_HFF_44C_120min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE105028_hESC_42C_60min_atacseq_hic_chromap_chipseq/atac_h3k27ac_intact_hic/encode_e2g_predictions_threshold0.259.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_12_5min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_25min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_30min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_30min_atacseq_GSE130758_K562_42C_30min_hic/atac_intact_hic/encode_e2g_predictions_threshold0.232.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_40min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_d21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_t21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE209675_MCF7_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE267931_HUVEC_39C_120min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe"),
#     pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE280307_THP1_42C_120min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.bedpe")
# ]

# for bedpe_path in bedpe_paths:
#     bedpe_path = pathlib.Path(bedpe_path)
#     out_path = bedpe_path.with_name(f"{bedpe_path.stem}_only_enhancer.bed")

#     bedpe = pl.read_csv(
#         bedpe_path,
#         separator="\t",
#         has_header=False,
#     )

#     enhancer_bed = bedpe.select(
#         pl.col("column_1"),
#         pl.col("column_2"),
#         pl.col("column_3"),
#     ).unique().sort("column_1", "column_2", "column_3")

#     enhancer_bed.write_csv(
#         out_path,
#         separator="\t",
#         include_header=False,
#     )
#     print(f"wrote: {out_path} ({enhancer_bed.height} intervals)")